# Feature Engineering

In [2]:
import pandas as pd
import logging

# ตั้งค่า Logging เบื้องต้น
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()

In [3]:
df_merged = pd.read_csv('data/bangkok_merge_congestion.csv')
df_weather = pd.read_csv('data/bangkok_hourly_weather_2024.csv')

In [4]:
df_merged

,ticket_id,type,organization,comment,district,timestamp,star,count_reopen,day,hour,month,year,lat,lon,resolution_time_hours,isLate,urgency_score,Day,congestion
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,ขยะเยอะ,บางซื่อ,2021-09-03 12:51:09.453003+00:00,0.0,0,Fri,12,9,2021,100.53084,13.81865,6578.718099,1,2,Fri,58
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ",น้ำท่วมเวลาฝนตกและทะลุเข้าบ้านเดือดร้อนมากทุกๆ...,ประเวศ,2021-09-19 14:56:08.924992+00:00,4.0,0,Sun,14,9,2021,100.66709,13.67891,6593.416835,1,5,Sun,40
2,2021-7XATFA,{สะพาน},เขตสาทร,สะพานลอยปรับปรุงไม่เสร็จตามกำหนด\nปากซอย สาทร12,สาทร,2021-09-26 05:03:52.594898+00:00,0.0,0,Sun,5,9,2021,100.52649,13.72060,6068.222133,1,5,Sun,0
3,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ",น้ำท่วม,บางซื่อ,2021-10-14 10:45:27.713884+00:00,0.0,0,Thu,10,10,2021,100.53099,13.81853,7893.837797,1,5,Thu,59
4,2021-DVEWYM,"{น้ำท่วม,ถนน}","เขตลาดพร้าว,ฝ่ายโยธา เขตลาดพร้าว",ซอยลาดพร้าววังหิน 75 ถนนลาดพร้าววังหิน แขวงลาด...,ลาดพร้าว,2021-12-09 12:29:08.408763+00:00,5.0,0,Thu,12,12,2021,100.59165,13.82280,5898.826799,1,5,Thu,54
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787020,KDDNBP,{ร้องเรียน},เขตหนองจอก,ศูนย์เรื่องราวร้องทุกข์ ได้รับการประสานผ่านระบ...,หนองจอก,2025-01-16 02:50:24.112172+00:00,0.0,0,Thu,2,1,2025,100.86304,13.85633,0.000741,0,1,Thu,0
787021,RYM99G,"{ความสะอาด,ถนน}","เขตคลองสาน,ฝ่ายสิ่งแวดล้อมฯ เขตคลองสาน",ปัญหา: บริเวณตลาดพระเครื่องพญาไม้ พบร้านอาหารต...,คลองสาน,2025-01-16 02:51:28.895700+00:00,0.0,0,Thu,2,1,2025,100.49651,13.73469,0.031729,0,4,Thu,0
787022,2025-BDQ87Y,"{ร้องเรียน,ต้นไม้,ความปลอดภัย}",เขตราชเทวี,ต้นไม้บริเวณดังกล่าวช่วงโคนต้นไม้มันแห้งเหมือน...,ราชเทวี,2025-01-16 02:52:33.878797+00:00,0.0,0,Thu,2,1,2025,100.53975,13.76426,0.001193,0,5,Thu,0
787023,EZCZZU,{ถนน},เขตจอมทอง,ปัญหา: พบโรงกลึง (ไม่ทราบชื่อ) ทำงานบนพื้นผิวจ...,จอมทอง,2025-01-16 02:53:27.583947+00:00,0.0,0,Thu,2,1,2025,100.46226,13.67672,0.000741,0,4,Thu,0


In [5]:
print(df_merged["timestamp"].dtype)

object


In [6]:
# --- Clean weather datetime ---
df_weather.columns = df_weather.columns.str.strip()
df_weather["datetime"] = pd.to_datetime(
    df_weather["datetime"].astype(str).str.strip(),
    errors="coerce"
)
df_merged["timestamp"] = pd.to_datetime(df_merged["timestamp"], errors="coerce")
# --- Fix timezone mismatch ---
# Traffy timestamp is timezone-aware, so remove timezone
df_merged["timestamp"] = df_merged["timestamp"].dt.tz_localize(None)

# Weather datetime is naive (no timezone), but do tz_localize(None) anyway to ensure consistency
df_weather["datetime"] = df_weather["datetime"].dt.tz_localize(None)

# --- Round timestamp to hour ---
df_merged["report_hour"] = df_merged["timestamp"].dt.floor("h")   # use "h", not "H"

# --- Merge by hour ---
df_merged = df_merged.merge(
    df_weather,
    how="left",
    left_on="report_hour",
    right_on="datetime"
)

logger.info(f"After merging weather: {df_merged.shape[0]} rows, {df_merged.shape[1]} columns")

df_merged[["timestamp", "report_hour", "datetime"]].head()

2025-12-07 22:31:10,359 - INFO - After merging weather: 787025 rows, 24 columns


,timestamp,report_hour,datetime
0,2021-09-03 12:51:09.453003,2021-09-03 12:00:00,NaT
1,2021-09-19 14:56:08.924992,2021-09-19 14:00:00,NaT
2,2021-09-26 05:03:52.594898,2021-09-26 05:00:00,NaT
3,2021-10-14 10:45:27.713884,2021-10-14 10:00:00,NaT
4,2021-12-09 12:29:08.408763,2021-12-09 12:00:00,NaT


In [7]:
# Change Column Name
rename = {
    "precipitation": "rain_volume",
    "temperature_2m": "temp",
    "wind_speed_10m": "wind_speed"
}
df_merged = df_merged.rename(columns={x: y for x, y in rename.items() if x in df_merged.columns})

df_merged['rain_volume'] = df_merged.get("rain_volume", 0)
df_merged['rain_volume'] = df_merged['rain_volume'].fillna(0)

# temp_std_6h: ค่าเบี่ยงเบนมาตรฐานของอุณหภูมิใน 6 ชม. ล่าสุด
# ถ้าค่าน้อย = อุณหภูมินิ่งๆ, ถ้าค่ามาก = อุณหภูมิสวิงขึ้นลงแรง
df_merged["temp_volatility_6h"] = df_merged["temp"].rolling(window=6).std()

# wind_max_min_diff: ความต่างของลมแรงสุดกับเบาสุดใน 3 ชม.
# บอกถึงความกระโชกของลม
wind_roll = df_merged["wind_speed"].rolling(window=3)
df_merged["wind_range_3h"] = wind_roll.max() - wind_roll.min()

# isTempVolatility: อุณหภูมิแกว่ง (Threshold 0.5 ถือว่าเริ่มมีการเปลี่ยนแปลงชัดเจน)
df_merged["isTempVolatility"] = (df_merged["temp_volatility_6h"] > 0.5).astype(int)

# isGustsWind: ลมกระโชก (แนะนำ Threshold ที่ 4.0 หรือ 5.0 m/s)
# เหตุผล: ความเร็วลมต่างกัน 4-5 m/s (ประมาณ 15-18 km/h) ในช่วงสั้นๆ คือลมวูบที่รู้สึกได้ชัดเจน
df_merged["isGustsWind"] = (df_merged["wind_range_3h"] > 4.0).astype(int)


In [8]:
df_merged['temp'].unique()

array([ nan, 27.2, 26.3, 25.6, 25. , 25.2, 25.4, 24.8, 25.8, 28.2, 30. ,
       31.4, 32.2, 32.8, 33.3, 33.1, 32.5, 30.9, 28.8, 27.9, 27.1, 26.5,
       26.1, 24.3, 24.1, 23.6, 23.7, 25.9, 30.3, 31.2, 32. , 32.6, 33. ,
       32.9, 31.7, 30.7, 27.6, 26.7, 26.2, 24.6, 24. , 23.8, 28. , 29.7,
       31.3, 32.3, 32.4, 31. , 30.1, 28.6, 23.5, 23.2, 23.3, 23. , 22.7,
       24.7, 29.2, 30.6, 31.5, 30.4, 27.8, 25.3, 24.2, 27.7, 29.5, 30.8,
       32.1, 29.4, 27. , 26.4, 24.4, 23.4, 31.8, 28.3, 26.9, 25.5, 23.1,
       22.9, 32.7, 30.5, 29.3, 27.5, 26.8, 25.7, 24.9, 23.9, 28.5, 30.2,
       31.9, 24.5, 25.1, 28.1, 31.6, 33.5, 34.1, 34. , 33.4, 29.6, 26.6,
       26. , 33.6, 29. , 27.4, 27.3, 28.4, 29.8, 28.7, 22.8, 22.4, 31.1,
       28.9, 22.5, 22.3, 29.1, 22.6, 22.2, 29.9, 33.2, 33.7, 33.8, 22. ,
       21.7, 20.8, 34.3, 34.8, 34.6, 33.9, 21.8, 21.5, 21.4, 34.4, 34.5,
       35. , 34.7, 35.5, 35.6, 34.9, 34.2, 35.2, 35.4, 35.1, 36.3, 37.4,
       37.8, 36.6, 35.3, 35.8, 35.7, 36.1, 36.4, 35

In [9]:
df_merged['temp_volatility_6h'].unique()

array([       nan, 0.4929503 , 0.464758  , ..., 0.36742346, 0.24494897,
       0.30983867])

In [10]:
df_merged[df_merged['isTempVolatility'] == 1].count()

ticket_id                9148
type                     9222
organization             9222
comment                  9148
district                 9219
timestamp                9222
star                     9222
count_reopen             9222
day                      9222
hour                     9222
month                    9222
year                     9222
lat                      9222
lon                      9222
resolution_time_hours    9222
isLate                   9222
urgency_score            9222
Day                      9222
congestion               9222
report_hour              9222
datetime                 9222
temp                     9222
rain_volume              9222
wind_speed               9222
temp_volatility_6h       9222
wind_range_3h            9222
isTempVolatility         9222
isGustsWind              9222
dtype: int64

In [11]:
print(f"Min timestamp in df_merged: {df_merged['timestamp'].min()}")
print(f"Max timestamp in df_merged: {df_merged['timestamp'].max()}")

Min timestamp in df_merged: 2021-09-03 12:51:09.453003
Max timestamp in df_merged: 2025-01-16 02:53:34.290375


In [12]:
# Define the date range
start_date = pd.to_datetime('2024-01-01')
end_date = pd.to_datetime('2024-12-31')

# Filter the DataFrame for the specified date range
records_in_range = df_merged[(df_merged['timestamp'] >= start_date) & (df_merged['timestamp'] <= end_date)]

# Count the number of records
num_records = len(records_in_range)

print(f"Number of records between {start_date.date()} and {end_date.date()}: {num_records}")

Number of records between 2024-01-01 and 2024-12-31: 307231


In [13]:
# Export เป็นไฟล์ CSV
records_in_range.to_csv('data_2024.csv', index=False, encoding='utf-8-sig')

print("Export successfully!")

Export successfully!


In [14]:
df_2024 = pd.read_csv('data_2024.csv')
print(f"Min timestamp in df_2024: {df_2024['timestamp'].min()}")
print(f"Max timestamp in df_2024: {df_2024['timestamp'].max()}")

Min timestamp in df_2024: 2024-01-01 00:07:06.679724
Max timestamp in df_2024: 2024-12-30 23:27:17.908622


In [15]:
df_2024['isGustsWind'].unique()

array([0, 1], dtype=int64)

In [16]:
feature_cols = [
    # categorical (จะ one-hot ในขั้น ML)
    "urgency_score",
    "organization",
    "district",

    # location
    "lat",
    "lon",

    # ticket behavior
    "star",
    "count_reopen",

    # time features
    "hour",
    "dayofweek",
    "month",
    "year",

    # weather features
    "rain_volume",
    "temp",
    "wind_speed",
    "isTempVolatility",
    "isGustsWind",

    # target
    "is_Late",
]

# เลือกเฉพาะคอลัมน์ที่มีอยู่จริงใน df (กันกรณีบาง column ไม่มี)
feature_cols_existing = [c for c in feature_cols if c in df_2024.columns]

df_features = df_2024[feature_cols_existing].copy()
logger.info(f"Final feature table shape: {df_features.shape}")
df_features.head()

2025-12-07 22:31:20,803 - INFO - Final feature table shape: (307231, 15)


,urgency_score,organization,district,lat,lon,star,count_reopen,hour,month,year,rain_volume,temp,wind_speed,isTempVolatility,isGustsWind
0,2,"เขตตลิ่งชัน,เขตบางกอกน้อย,ฝ่ายสิ่งแวดล้อมฯ เขต...",ตลิ่งชัน,100.46033,13.75419,5.0,0,0,1,2024,0.0,27.2,10.9,0,0
1,3,"เขตวังทองหลาง,ฝ่ายสิ่งแวดล้อมฯ เขตวังทองหลาง,ก...",วังทองหลาง,100.61698,13.78799,5.0,0,0,1,2024,0.0,27.2,10.9,0,0
2,1,"เขตราชเทวี,ฝ่ายเทศกิจ เขตราชเทวี",ราชเทวี,100.52951,13.75178,0.0,0,0,1,2024,0.0,27.2,10.9,0,0
3,2,"เขตสวนหลวง,สำนักสิ่งแวดล้อม กทม.,ฝ่ายสิ่งแวดล้...",สวนหลวง,100.60225,13.71287,0.0,0,1,1,2024,0.0,26.3,10.5,0,0
4,1,"เขตบางแค,สน.เพชรเกษม,สำนักงานตำรวจแห่งชาติ (Ro...",บางแค,100.41546,13.68437,1.0,0,1,1,2024,0.0,26.3,10.5,0,0


In [17]:
df_features

,urgency_score,organization,district,lat,lon,star,count_reopen,hour,month,year,rain_volume,temp,wind_speed,isTempVolatility,isGustsWind
0,2,"เขตตลิ่งชัน,เขตบางกอกน้อย,ฝ่ายสิ่งแวดล้อมฯ เขต...",ตลิ่งชัน,100.46033,13.75419,5.0,0,0,1,2024,0.0,27.2,10.9,0,0
1,3,"เขตวังทองหลาง,ฝ่ายสิ่งแวดล้อมฯ เขตวังทองหลาง,ก...",วังทองหลาง,100.61698,13.78799,5.0,0,0,1,2024,0.0,27.2,10.9,0,0
2,1,"เขตราชเทวี,ฝ่ายเทศกิจ เขตราชเทวี",ราชเทวี,100.52951,13.75178,0.0,0,0,1,2024,0.0,27.2,10.9,0,0
3,2,"เขตสวนหลวง,สำนักสิ่งแวดล้อม กทม.,ฝ่ายสิ่งแวดล้...",สวนหลวง,100.60225,13.71287,0.0,0,1,1,2024,0.0,26.3,10.5,0,0
4,1,"เขตบางแค,สน.เพชรเกษม,สำนักงานตำรวจแห่งชาติ (Ro...",บางแค,100.41546,13.68437,1.0,0,1,1,2024,0.0,26.3,10.5,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307226,3,"เขตดุสิต,เขตพระนคร,สำนักงานเลขานุการปลัดกรุงเท...",พระนคร,100.50125,13.75241,3.0,0,22,12,2024,0.0,27.0,4.4,0,0
307227,4,"เขตบางขุนเทียน,สำนักการโยธา กทม.,สำนักงานก่อสร...",บางขุนเทียน,100.38820,13.58846,0.0,0,22,12,2024,0.0,27.0,4.4,0,0
307228,4,"เขตบางขุนเทียน,สำนักการโยธา กทม.,สำนักงานก่อสร...",บางขุนเทียน,100.38846,13.58991,0.0,0,22,12,2024,0.0,27.0,4.4,0,0
307229,4,"เขตบางขุนเทียน,สำนักการโยธา กทม.,สำนักงานก่อสร...",บางขุนเทียน,100.38841,13.58355,0.0,0,22,12,2024,0.0,27.0,4.4,0,0


In [19]:
df_features.columns

Index(['urgency_score', 'organization', 'district', 'lat', 'lon', 'star',
       'count_reopen', 'hour', 'month', 'year', 'rain_volume', 'temp',
       'wind_speed', 'isTempVolatility', 'isGustsWind'],
      dtype='object')

In [18]:
df_features.sample(5)

,urgency_score,organization,district,lat,lon,star,count_reopen,hour,month,year,rain_volume,temp,wind_speed,isTempVolatility,isGustsWind
70022,3,"เขตปทุมวัน,ฝ่ายเทศกิจ เขตปทุมวัน",ปทุมวัน,100.52566,13.73437,3.0,0,9,3,2024,0.0,31.1,11.2,1,0
275910,4,"เขตบางเขน,ฝ่ายเทศกิจ เขตบางเขน",บางเขน,100.67882,13.86853,5.0,0,12,11,2024,0.0,32.4,6.6,0,0
57319,2,"เขตหลักสี่,ฝ่ายสิ่งแวดล้อมฯ เขตหลักสี่,สำนักงา...",หลักสี่,100.58015,13.86804,5.0,0,4,3,2024,0.0,27.8,10.8,0,0
276053,2,"เขตบางนา,สน.บางนา,สำนักงานตำรวจแห่งชาติ (Royal...",บางนา,100.60773,13.67575,0.0,0,16,11,2024,0.1,32.9,10.2,0,0
126833,1,"เขตทุ่งครุ,สำนักสิ่งแวดล้อม กทม.,ฝ่ายเทศกิจ เข...",ทุ่งครุ,100.47904,13.65990,5.0,0,0,6,2024,0.0,27.3,10.0,0,0
